# Consultor Financeiro Inteligente - Quantum Finance

Os agentes (LangGraph) falam apenas HTTP com o **BFA** (Backend For Agents), que é o único dono da sessão MCP do Bolsai.

Antes de rodar: suba o BFA em outro terminal com `uv run python -m bfa` e preencha o `.env` (veja `.env.example`).

## 1. Instalação (opcional)
O ambiente é gerenciado pelo `uv`. Basta rodar `uv sync` na pasta do projeto.

In [ ]:
# Instalação das dependências (já feita pelo 'uv sync'; descomente só se necessário)
#!uv sync

## 2. Credenciais e caminhos

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Garante que a raiz do projeto está no sys.path (para importar o pacote 'agent')
raiz = Path.cwd()
if not (raiz / 'agent').exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

load_dotenv(raiz / '.env')
print('OPENAI_API_KEY definida:', bool(os.environ.get('OPENAI_API_KEY')))

## 3. Saúde do BFA

In [ ]:
from agent.bfa_client import BFAClient

bfa = BFAClient()
print(bfa.health())

## 4. Grafo de agentes

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

from agent.graph import build_graph

app = build_graph()  # Lead Advisor -> Market Analyst / Agente B3, com memória (MemorySaver)
print('Grafo compilado com sucesso!')

## 5. Chat interativo

In [ ]:
def start_chat(perfil: dict):
    """Inicia um chat conversacional interativo com histórico de mensagens usando o checkpointer do LangGraph"""
    print("=== INICIANDO CONSULTORIA FINANCEIRA INTERATIVA ===")
    print(f"Perfil do Cliente: {perfil}")
    print("Digite 'sair' para encerrar a conversa.\n")

    config = {"configurable": {"thread_id": "consultoria_client_thread"}, "recursion_limit": 15}

    while True:
        try:
            pergunta = input("Você: ")
        except KeyboardInterrupt:
            print("\nSaindo...")
            break

        if pergunta.lower().strip() in ["sair", "exit", "quit"]:
            print("Consultoria finalizada. Até mais!")
            break

        if not pergunta.strip():
            continue

        print("\n[Consultando agentes e processando resposta...]")

        for event in app.stream({"messages": [HumanMessage(content=pergunta)], "user_profile": perfil}, config):
            for node_name, state_update in event.items():
                for msg in state_update.get("messages", []):
                    if isinstance(msg, AIMessage):
                        if msg.tool_calls:
                            for tc in msg.tool_calls:
                                print(f" -> [Estrategista] Delegando: {tc['name']} com parâmetros {tc['args']}")
                        else:
                            print(f"\n[Relatório Final]:\n{msg.content}\n")
                    elif isinstance(msg, ToolMessage):
                        print(" -> [Subagente] Retornou resposta.")

In [ ]:
# Caso de Teste: Gabriel, investidor Arrojado buscando alocação de longo prazo e cotações
perfil_arrojado = {
    "Nome": "Gabriel",
    "Perfil de Risco": "Arrojado",
    "Objetivo": "Maximizar ganhos em renda variável no longo prazo",
    "Horizonte de Tempo": "10 anos"
}

# Chat conversacional
start_chat(perfil=perfil_arrojado)

## 6. Exemplo de chamada direta ao BFA
O mesmo serviço que os agentes usam pode ser consultado diretamente.

In [ ]:
# Snapshot composto (cotação + fundamentos + dividendos) de uma ação
print(BFAClient.as_llm_text(bfa.snapshot('WEGE3')))

# Indicador macro (o alias 'usdbrl' é aceito e normalizado para 'usd_brl')
print(BFAClient.as_llm_text(bfa.macro('usdbrl', limit=5)))